# EN4554 — Assignment 2: Convolutional Neural Networks

| | |
|---|---|
| **Name** | *Bandara RMTD* |
| **Index number** | *220067G* |
| **GitHub repository** | *https://github.com/your-username/en4554-a2* |

**Acknowledgement of AI tool use.** *(Edit this so that it is accurate.)* I used Claude (Anthropic) to help write the code and to draft the structure of this notebook. I have read, run and understood every cell, tuned the settings myself where stated, and checked every interpretation against the results that I obtained.

**Contents**  

0. Setup: data split, normalisation and shared training utilities  
1. Backpropagation from scratch (layers, gradient checks, MLP training, vanishing gradients)  
2. Optimizers (SGD with momentum and Adam from scratch, a 2-D trajectory study, learning-rate sensitivity on a CNN)  
3. Regularisation (7 settings on 5k images) and residual connections (plain vs residual networks of depth 8, 20, 56)  
4. Best CNN with at most 500k parameters and 20 epochs, with ablations and a single test-set evaluation  
5. Conclusions  

**How to run.** On Colab choose *Runtime → Change runtime type → T4 GPU*, then *Run all*. The whole notebook takes roughly 1–1.5 hours on a T4 (most of it in Tasks 3(b) and 4). To check that everything works first, set `FAST_DEV_RUN = True` in the first code cell: every task then runs on tiny subsets for 2 epochs in a few minutes. Training results are cached in `results/`, so re-running a cell after a disconnect does not repeat the training (delete a file to force a re-run).

**Placeholders.** Each interpretation cell explains the mechanism behind the expected result. Numbers written as **[??]** must be filled in from your own run, and any statement that your plots do not support must be corrected.

## 0. Setup

In [1]:
# ---------------------------------------------------------------
# Imports and global configuration
# ---------------------------------------------------------------
import os, re, math, time, copy, random, pickle
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
DATA_ROOT = './data'
# Training results are cached here so that re-running a cell (or recovering from a Colab disconnect) does not
# repeat the training. Delete a file to force that run to be repeated. On Colab you can mount Google Drive and
# point this at e.g. '/content/drive/MyDrive/en4554_a2_results' so the cache survives a runtime reset.
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def pick_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')        # Apple-silicon laptops
    return torch.device('cpu')

device = pick_device()
USE_AMP = device.type == 'cuda'           # float16 mixed precision on NVIDIA GPUs only (about 2x faster on a T4)
torch.backends.cudnn.benchmark = True     # cuDNN picks the fastest conv algorithms (tiny run-to-run noise on GPU)

def seed_everything(seed=SEED):
    """Fix every random number generator so that runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)               # seeds the CPU and all CUDA devices


seed_everything()
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
gpu_name = f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''
print(f'PyTorch {torch.__version__} | device: {device}{gpu_name} | mixed precision: {USE_AMP}')

PyTorch 2.6.0+cu118 | device: cuda (NVIDIA GeForce GTX 1650) | mixed precision: True


### Data

The 50,000 CIFAR-10 training images are split once, with a fixed seed, into **45,000 training** and **5,000 validation** images. The per-channel mean and standard deviation are computed on the 45k training images only and are then used to normalise the training, validation and (in Task 4) test images, so that no information from the held-out data leaks into preprocessing. The **test set is not loaded until Task 4**, where it is used exactly once.

The normalised images are kept as tensors on the GPU (about 0.6 GB) and mini-batches are cut directly from them, and data augmentation is also done on the GPU. This avoids the slow CPU `DataLoader` pipeline on Colab's two CPU cores.

In [4]:
# ---------------------------------------------------------------
# CIFAR-10: 45k train / 5k validation split, per-channel normalisation with TRAINING statistics
# ---------------------------------------------------------------
cifar_train = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
CLASSES = cifar_train.classes

X_all = torch.tensor(cifar_train.data).permute(0, 3, 1, 2).float().div_(255)   # (50000, 3, 32, 32) in [0, 1]
Y_all = torch.tensor(cifar_train.targets, dtype=torch.long)

split_gen = torch.Generator().manual_seed(SEED)               # fixed seed -> the same split every time
perm = torch.randperm(len(X_all), generator=split_gen)
train_idx, val_idx = perm[:45_000], perm[45_000:]
X_tr, Y_tr = X_all[train_idx], Y_all[train_idx]
X_va, Y_va = X_all[val_idx], Y_all[val_idx]
del X_all, cifar_train

MEAN = X_tr.mean(dim=(0, 2, 3))                               # statistics of the 45k TRAINING images only
STD = X_tr.std(dim=(0, 2, 3))

def normalize(x):
    """Per-channel standardisation with the training-set mean and std (used for train, val and test)."""
    return (x - MEAN.view(1, 3, 1, 1)) / STD.view(1, 3, 1, 1)

# normalise in place to save memory, then move everything to the device
X_train = X_tr.sub_(MEAN.view(1, 3, 1, 1)).div_(STD.view(1, 3, 1, 1)).to(device)
X_val = X_va.sub_(MEAN.view(1, 3, 1, 1)).div_(STD.view(1, 3, 1, 1)).to(device)
Y_train, Y_val = Y_tr.to(device), Y_va.to(device)
del X_tr, X_va, Y_tr, Y_va


print(f'train {tuple(X_train.shape)}   validation {tuple(X_val.shape)}')
print('training mean per channel:', MEAN.numpy().round(4), '  std:', STD.numpy().round(4))
print('after normalisation (train): mean', X_train.mean(dim=(0, 2, 3)).cpu().numpy().round(3),
      ' std', X_train.std(dim=(0, 2, 3)).cpu().numpy().round(3))
print('validation class counts:', torch.bincount(Y_val, minlength=10).tolist())

  8%|▊         | 12.9M/170M [04:35<56:06, 46.8kB/s]  


KeyboardInterrupt: 

In [ ]:
def show_rows(rows, n=8):
    """rows: list of (title, images, labels). Undoes the normalisation and plots n images per row."""
    fig, axes = plt.subplots(len(rows), n, figsize=(1.5 * n, 1.75 * len(rows)), squeeze=False)
    for r, (title, X, Y) in enumerate(rows):
        imgs = (X[:n].detach().float().cpu() * STD.view(1, 3, 1, 1) + MEAN.view(1, 3, 1, 1)).clamp(0, 1)
        for c in range(n):
            ax = axes[r, c]
            ax.imshow(imgs[c].permute(1, 2, 0).numpy())
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            ax.set_title(CLASSES[int(Y[c])], fontsize=8)
        axes[r, 0].set_ylabel(title, fontsize=8)
    plt.tight_layout(); plt.show()

show_rows([('training images', X_train, Y_train)], n=10)

### Shared utilities

These helpers are used by every task from Task 2 onwards:

* `batches` cuts shuffled mini-batches from the on-device tensors.
* `random_flip`, `random_crop`, `cutout` and `mixup` are my own vectorised GPU implementations of the augmentations (Task 3 requires cutout and mixup to be implemented by hand).
* `evaluate` returns the mean cross-entropy and accuracy of a model on a dataset.
* `train` is the generic training loop. It supports augmentation, mixup, label smoothing, a learning-rate schedule and mixed precision, and it records the loss of every iteration as well as the per-epoch training loss, validation loss and accuracy (and optionally the accuracy on clean training images).
* `cached` stores the result of a training run on disk and reloads it the next time.

In [ ]:
# ---------------------------------------------------------------
# Mini-batches and data augmentation (all on the device, vectorised over the batch)
# ---------------------------------------------------------------
def batches(X, Y, batch_size, shuffle=False, drop_last=False):
    """Yield (x, y) mini-batches from tensors that already live on the device."""
    N = X.shape[0]
    idx = torch.randperm(N, device=X.device) if shuffle else torch.arange(N, device=X.device)
    stop = N - N % batch_size if (drop_last and N >= batch_size) else N
    for k in range(0, stop, batch_size):
        j = idx[k:k + batch_size]
        yield X[j], Y[j]

def random_flip(x):
    """Horizontal flip of each image with probability 1/2."""
    flip = torch.rand(x.shape[0], device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)

def random_crop(x, pad=4):
    """Zero-pad by `pad` pixels and take a random 32x32 crop per image (a random shift of up to +-pad pixels).
    Zero is the mean colour after normalisation."""
    B, C, H, W = x.shape
    xp = F.pad(x, (pad, pad, pad, pad))
    i = torch.randint(0, 2 * pad + 1, (B,), device=x.device)                 # top offset of each crop
    j = torch.randint(0, 2 * pad + 1, (B,), device=x.device)                 # left offset of each crop
    rows = (i[:, None] + torch.arange(H, device=x.device))[:, :, None]       # (B, H, 1)
    cols = (j[:, None] + torch.arange(W, device=x.device))[:, None, :]       # (B, 1, W)
    b = torch.arange(B, device=x.device)[:, None, None]                      # (B, 1, 1)
    return xp.permute(0, 2, 3, 1)[b, rows, cols].permute(0, 3, 1, 2).contiguous()

def cutout(x, size=16):
    """Cutout (DeVries & Taylor, 2017): zero a size x size square at a random centre in every image.
    The square may extend beyond the border (then a smaller area is erased), as in the original paper."""
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B, 1, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1, 1), device=x.device)
    ys = torch.arange(H, device=x.device)[None, :, None]
    xs = torch.arange(W, device=x.device)[None, None, :]
    inside = (ys >= cy - size // 2) & (ys < cy + size - size // 2) & (xs >= cx - size // 2) & (xs < cx + size - size // 2)
    return x * (~inside)[:, None].to(x.dtype)

def mixup(x, y, alpha):
    """Mixup (Zhang et al., 2018): convex combination of the batch with a shuffled copy of itself.
    Returns the mixed images, both label vectors and the mixing weight lambda ~ Beta(alpha, alpha)."""
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.shape[0], device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

def make_augment(flip_crop=False, cutout_size=0):
    """Build the per-batch augmentation function (None if no augmentation)."""
    if not flip_crop and not cutout_size:
        return None
    def augment(x):
        if flip_crop:
            x = random_crop(random_flip(x))
        if cutout_size:
            x = cutout(x, cutout_size)
        return x
    return augment

In [ ]:
# ---------------------------------------------------------------
# Evaluation, training loop, learning-rate schedule, caching
# ---------------------------------------------------------------
def autocast_ctx():
    return torch.autocast('cuda', dtype=torch.float16) if USE_AMP else nullcontext()

def make_grad_scaler():
    return torch.amp.GradScaler('cuda', enabled=USE_AMP)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def evaluate(model, X, Y, batch_size=1000):
    """Mean cross-entropy and accuracy of `model` on (X, Y), in eval mode (BN uses running stats, no dropout)."""
    model.eval()
    total_loss, correct = 0.0, 0
    for xb, yb in batches(X, Y, batch_size):
        with autocast_ctx():
            out = model(xb)
        total_loss += F.cross_entropy(out.float(), yb, reduction='sum').item()
        correct += (out.argmax(1) == yb).sum().item()
    return total_loss / len(X), correct / len(X)

def cosine_schedule(warmup_frac=0.0):
    """Learning-rate multiplier: linear warm-up over the first warmup_frac of the steps, then cosine decay to 0."""
    def schedule(step, total):
        warm = int(warmup_frac * total)
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    return schedule

def train(model, optimizer, n_epochs, X, Y, X_val, Y_val, batch_size=128, augment=None, mixup_alpha=0.0,
          label_smoothing=0.0, lr_schedule=None, eval_train=None, name='', print_every=None):
    """Generic mini-batch training loop (Tasks 2-4).
      augment         function applied to every training batch, or None
      mixup_alpha     > 0 enables mixup with lambda ~ Beta(alpha, alpha)
      lr_schedule     function(step, total_steps) -> multiplier of the base learning rate (None = constant)
      eval_train      (X, Y) of CLEAN training images; if given, the training accuracy is measured on them
                      in eval mode at the end of every epoch
    Returns a dict with per-epoch histories and the loss of every iteration."""
    scaler = make_grad_scaler()
    steps_per_epoch = max(1, len(X) // batch_size)
    total_steps = n_epochs * steps_per_epoch
    base_lrs = [g['lr'] for g in optimizer.param_groups]
    print_every = print_every or max(1, n_epochs // 10)
    hist = dict(train_loss=[], val_loss=[], val_acc=[], train_acc=[], iter_loss=[],
                steps_per_epoch=steps_per_epoch, diverged=False)
    step, t_start = 0, time.time()
    for ep in range(n_epochs):
        model.train()
        ep_losses = []
        for xb, yb in batches(X, Y, batch_size, shuffle=True, drop_last=True):
            if lr_schedule is not None:
                for g, lr0 in zip(optimizer.param_groups, base_lrs):
                    g['lr'] = lr0 * lr_schedule(step, total_steps)
            if augment is not None:
                xb = augment(xb)
            if mixup_alpha > 0:
                xb, y_a, y_b, lam = mixup(xb, yb, mixup_alpha)
            with autocast_ctx():
                out = model(xb)
                if mixup_alpha > 0:   # the loss is mixed with the same weight as the images
                    loss = lam * F.cross_entropy(out, y_a, label_smoothing=label_smoothing) \
                           + (1 - lam) * F.cross_entropy(out, y_b, label_smoothing=label_smoothing)
                else:
                    loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ep_losses.append(loss.detach())
            step += 1
        ep_losses = torch.stack(ep_losses).float().cpu()       # one GPU->CPU transfer per epoch
        hist['iter_loss'] += ep_losses.tolist()
        hist['train_loss'].append(ep_losses.mean().item())
        vl, va = evaluate(model, X_val, Y_val)
        hist['val_loss'].append(vl)
        hist['val_acc'].append(va)
        if eval_train is not None:
            hist['train_acc'].append(evaluate(model, *eval_train)[1])
        diverged = not bool(torch.isfinite(ep_losses).all())
        if (ep + 1) % print_every == 0 or ep == 0 or ep == n_epochs - 1 or diverged:
            msg = (f'{name:>26s} | ep {ep + 1:3d}/{n_epochs} | train loss {hist["train_loss"][-1]:.4f} '
                   f'| val loss {vl:.4f} | val acc {100 * va:5.2f}%')
            if eval_train is not None:
                msg += f' | train acc {100 * hist["train_acc"][-1]:5.2f}%'
            print(msg + f' | {time.time() - t_start:6.1f}s')
        if diverged:
            print(f'{name:>26s} | the loss became NaN/inf: the run diverged, stopping it')
            hist['diverged'] = True
            break
    hist['time'] = time.time() - t_start
    return hist

def slug(s):
    return re.sub(r'[^A-Za-z0-9.+-]+', '_', s).strip('_')

def cached(key, fn):
    """Return the stored result of run `key` if it exists, otherwise compute fn(), store it and return it."""
    path = os.path.join(RESULTS_DIR, slug(key) + '.pkl')
    if os.path.exists(path):
        print(f'[cache] loaded {path}  (delete it to re-run)')
        with open(path, 'rb') as f:
            return pickle.load(f)
    out = fn()
    with open(path, 'wb') as f:
        pickle.dump(out, f)
    return out

def plot_smoothed_loss(ax, hist, label, k=50, **kw):
    """Plot the per-iteration training loss, smoothed with a moving average of k iterations, against epochs."""
    y = np.asarray(hist['iter_loss'], dtype=float)
    k = max(1, min(k, len(y) // 5))
    ys = np.convolve(y, np.ones(k) / k, mode='valid')
    xs = (np.arange(len(ys)) + k) / hist['steps_per_epoch']
    ax.plot(xs, ys, label=label, **kw)

## Task 1 — Backpropagation

### 1(a) Linear, ReLU and softmax cross-entropy with explicit forward and backward passes

For a mini-batch of $N$ row vectors stacked in $X \in \mathbb{R}^{N\times d}$, and with $G = \partial L/\partial(\text{output})$ the gradient arriving from the layer above, the layers and their backward passes are:

* **Linear**, $Y = XW + b$ with $W\in\mathbb{R}^{d\times m}$:
$\quad \dfrac{\partial L}{\partial W} = X^\top G,\qquad \dfrac{\partial L}{\partial b} = \mathbf{1}^\top G \;(\text{sum over the batch}),\qquad \dfrac{\partial L}{\partial X} = G\,W^\top.$
* **ReLU**, $Y = \max(0, X)$: $\quad \dfrac{\partial L}{\partial X} = G \odot \mathbb{1}[X>0]$.
* **Sigmoid** (needed in 1(d)), $Y = \sigma(X)$: $\quad \dfrac{\partial L}{\partial X} = G \odot \sigma(X)\,(1-\sigma(X))$.
* **Softmax cross-entropy** on logits $Z$ with labels $y$: $p_{ik} = \dfrac{e^{z_{ik}}}{\sum_j e^{z_{ij}}}$, $\;L = -\dfrac1N\sum_i \log p_{i,y_i}$, and
$\quad \dfrac{\partial L}{\partial Z} = \dfrac{1}{N}\,\big(P - \mathrm{onehot}(y)\big).$

The softmax is computed with the log-sum-exp trick (subtract the row maximum before exponentiating), which leaves the result unchanged but avoids overflow. Each layer caches what its backward pass needs (the input of `Linear`, the mask of `ReLU`, the output of `Sigmoid`, the probabilities of the loss). `MLP.backward` applies the chain rule by calling the layers' `backward` methods in reverse order. No tensor in this task has `requires_grad=True`, so autograd is never used.

In [ ]:
class Linear:
    """Fully connected layer  y = x W + b  with a hand-written backward pass."""
    def __init__(self, in_features, out_features, init='kaiming', dtype=torch.float32, device=device):
        if init == 'kaiming':     # He et al. (2015): Var(W) = 2/fan_in keeps the signal variance constant through ReLUs
            std = math.sqrt(2.0 / in_features)
        elif init == 'xavier':    # Glorot & Bengio (2010): Var(W) = 2/(fan_in + fan_out), for symmetric activations
            std = math.sqrt(2.0 / (in_features + out_features))
        else:
            raise ValueError(init)
        self.W = torch.randn(in_features, out_features, dtype=dtype, device=device) * std
        self.b = torch.zeros(out_features, dtype=dtype, device=device)
        self.dW, self.db = torch.zeros_like(self.W), torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x                        # cached: needed for dL/dW
        return x @ self.W + self.b

    def backward(self, grad_out):         # grad_out = dL/dy, shape (N, out)
        self.dW = self.x.T @ grad_out     # (in, out)
        self.db = grad_out.sum(dim=0)     # (out,)
        return grad_out @ self.W.T        # dL/dx, shape (N, in)


class ReLU:
    def forward(self, x):
        self.mask = x > 0                 # cached: where the gradient passes through
        return x.clamp(min=0)

    def backward(self, grad_out):
        return grad_out * self.mask


class Sigmoid:
    def forward(self, x):
        self.s = torch.sigmoid(x)         # cached: sigma'(x) = s (1 - s)
        return self.s

    def backward(self, grad_out):
        return grad_out * self.s * (1 - self.s)


class SoftmaxCrossEntropy:
    """Mean cross-entropy of softmax(logits) against integer labels."""
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values          # log-sum-exp trick for numerical stability
        log_p = z - z.exp().sum(dim=1, keepdim=True).log()
        self.p, self.y = log_p.exp(), y                               # cached for the backward pass
        return -log_p[torch.arange(len(y), device=y.device), y].mean()

    def backward(self):
        N = self.y.shape[0]
        grad = self.p.clone()
        grad[torch.arange(N, device=self.y.device), self.y] -= 1      # P - onehot(y)
        return grad / N


class MLP:
    """Multi-layer perceptron built from the layers above: Linear -> act -> ... -> Linear -> softmax CE."""
    def __init__(self, sizes, activation='relu', init='kaiming', dtype=torch.float32, device=device):
        self.layers = []
        for i, (d_in, d_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(d_in, d_out, init, dtype, device))
            if i < len(sizes) - 2:                                    # no activation after the last layer
                self.layers.append(ReLU() if activation == 'relu' else Sigmoid())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x                                                      # logits

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        grad = self.loss_fn.backward()
        for layer in reversed(self.layers):                           # chain rule, from the output back
            grad = layer.backward(grad)
        return grad                                                   # dL/d(input)

    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def parameters(self):
        return [p for l in self.linears() for p in (l.W, l.b)]

    def gradients(self):
        return [g for l in self.linears() for g in (l.dW, l.db)]

    def sgd_step(self, lr):
        for l in self.linears():
            l.W -= lr * l.dW
            l.b -= lr * l.db

# quick sanity check: an untrained network should give a loss of the order of ln(10) = 2.303 (uniform guessing).
# It comes out somewhat higher because He initialisation of the last layer gives logits of order-one variance,
# i.e. random but already fairly confident predictions.
_net = MLP([3072, 512, 256, 10])
print('logits shape:', tuple(_net.forward(X_train[:256].flatten(1)).shape),
      '| initial loss:', round(_net.loss(X_train[:256].flatten(1), Y_train[:256]).item(), 3), '| ln 10 =', round(math.log(10), 3))

### 1(b) Gradient checks against autograd and against centred finite differences

The hand-written gradients are compared with two independent references:

1. **Autograd.** The same network, with copies of the same parameters, is evaluated with ordinary PyTorch operations (`@`, `torch.relu`, `F.cross_entropy`) and differentiated with `loss.backward()`.
2. **Centred finite differences.** For a parameter entry $\theta_k$: $\;\dfrac{\partial L}{\partial \theta_k} \approx \dfrac{L(\theta + h e_k) - L(\theta - h e_k)}{2h}$, with error $O(h^2)$. Each entry needs two forward passes, so 25 randomly chosen entries of every tensor are checked.

The agreement is measured with the relative error $\;\mathrm{rel}(a,b) = \dfrac{\lVert a-b\rVert_2}{\lVert a\rVert_2 + \lVert b\rVert_2}$ (0 means identical, values near 1 mean unrelated) and with `torch.allclose`. The check runs in **float64** on a small MLP (3072–64–32–10) and 16 real CIFAR-10 images. Double precision is essential for finite differences, whose round-off error is about $\varepsilon_{\text{mach}}\,|L|/h$: with $h=10^{-6}$ this is about $10^{-10}$ in float64 but would be about $10^{-1}$ in float32. The biases are set to random values so that their gradients are tested in a non-trivial state, and the gradient with respect to the input is checked too. Both activations are tested because Task 1(d) uses the sigmoid network.

Tolerances: autograd computes the same mathematics in a different order, so it must agree to round-off (`rtol=1e-7, atol=1e-10`). Finite differences are approximate, so a looser tolerance is used (`rtol=1e-5, atol=1e-7`).

In [ ]:
def rel_error(a, b):
    """||a - b|| / (||a|| + ||b||)."""
    return ((a - b).norm() / (a.norm() + b.norm()).clamp_min(1e-300)).item()

def autograd_reference(net, x, y):
    """Run the same network with PyTorch autograd on copies of the same parameters."""
    params = [p.clone().requires_grad_(True) for p in net.parameters()]
    x = x.clone().requires_grad_(True)
    h, k = x, 0
    for layer in net.layers:
        if isinstance(layer, Linear):
            h = h @ params[k] + params[k + 1]
            k += 2
        elif isinstance(layer, ReLU):
            h = torch.relu(h)
        else:
            h = torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return loss.item(), [p.grad for p in params], x.grad

def finite_difference(net, x, y, tensor, n_checks=25, h=1e-6, gen=None):
    """Centred differences for n_checks random entries of `tensor` (perturbed in place, then restored)."""
    flat = tensor.view(-1)
    idx = torch.randperm(flat.numel(), generator=gen)[:n_checks]
    numeric = torch.empty(len(idx), dtype=tensor.dtype)
    for k, i in enumerate(idx):
        old = flat[i].item()
        flat[i] = old + h; loss_plus = net.loss(x, y).item()
        flat[i] = old - h; loss_minus = net.loss(x, y).item()
        flat[i] = old
        numeric[k] = (loss_plus - loss_minus) / (2 * h)
    return idx, numeric

rows = []
fd_gen = torch.Generator().manual_seed(SEED)
for act in ['relu', 'sigmoid']:
    torch.manual_seed(SEED)
    net = MLP([3072, 64, 32, 10], activation=act, dtype=torch.float64, device=torch.device('cpu'))
    for lin in net.linears():
        lin.b.normal_(0, 0.1)                                   # non-zero biases
    x = X_train[:16].flatten(1).double().cpu()
    y = Y_train[:16].cpu()

    loss_mine = net.loss(x, y).item()                          # my forward pass
    dx_mine = net.backward()                                   # my backward pass
    grads_mine = [g.clone() for g in net.gradients()] + [dx_mine]
    loss_ref, grads_ref, dx_ref = autograd_reference(net, x, y)
    grads_ref = grads_ref + [dx_ref]
    print(f'{act:>7s}: loss (mine) = {loss_mine:.12f}   loss (autograd) = {loss_ref:.12f}')

    names = [f'{n}{i + 1}' for i in range(len(net.linears())) for n in ('W', 'b')] + ['input x']
    tensors = net.parameters() + [x]
    for name, t, g_mine, g_ref in zip(names, tensors, grads_mine, grads_ref):
        idx, g_num = finite_difference(net, x, y, t, gen=fd_gen)
        g_ana = g_mine.reshape(-1)[idx]
        rows.append({'activation': act, 'tensor': name, 'shape': str(tuple(t.shape)),
                     'rel. err. vs autograd': rel_error(g_mine, g_ref),
                     'allclose (autograd)': torch.allclose(g_mine, g_ref, rtol=1e-7, atol=1e-10),
                     'rel. err. vs finite diff.': rel_error(g_ana, g_num),
                     'allclose (finite diff.)': torch.allclose(g_ana, g_num, rtol=1e-5, atol=1e-7)})

df_check = pd.DataFrame(rows)
for col in ['rel. err. vs autograd', 'rel. err. vs finite diff.']:
    df_check[col] = df_check[col].map(lambda v: f'{v:.2e}')
display(df_check)
print('All checks pass:', bool(df_check['allclose (autograd)'].all() and df_check['allclose (finite diff.)'].all()))

**Interpretation (1b).** ✏️ For both activations, every parameter tensor and the input gradient agree with autograd to a relative error of about **[??]** (of order $10^{-16}$, i.e. machine precision in float64), and `torch.allclose` returns `True` everywhere. Since autograd evaluates the same mathematical function, any remaining difference is only floating-point round-off from performing the operations in a different order. The two loss values also agree to all printed digits, which confirms that the forward passes are identical too.

The relative errors against centred finite differences are larger, about **[??]** (of order $10^{-9}$ to $10^{-7}$). This does not indicate a bug: the finite-difference estimate is itself approximate, with a truncation error of $O(h^2)$ and a round-off error of $O(\varepsilon_{\text{mach}}/h)$, and both are around $10^{-10}$ in absolute terms for $h=10^{-6}$. A wrong backward pass, for example a missing transpose or a forgotten $1/N$, would give relative errors of order $10^{-1}$ to $1$. The finite-difference check is the more fundamental of the two because it does not depend on any other implementation of the same formulas. Together, the two checks show that the explicit backward passes of `Linear`, `ReLU`, `Sigmoid` and softmax cross-entropy are correct.

### 1(c) Training the MLP with SGD on flattened images

Each image is flattened to a vector of length $3\cdot32\cdot32 = 3072$. The network is 3072–512–256–10 with ReLU activations and He initialisation (1.71M parameters). It is trained with plain mini-batch SGD ($\theta \leftarrow \theta - \eta\,\nabla_\theta L$) with a constant learning rate $\eta = 0.05$, batch size 128 and 20 epochs, using only the hand-written forward and backward passes. After every epoch the loss and accuracy on the full training and validation sets are recorded.

In [ ]:
seed_everything()
mlp = MLP([3072, 512, 256, 10], activation='relu', init='kaiming')
Xtr_flat, Xval_flat = X_train.flatten(1), X_val.flatten(1)          # (N, 3072) views, no copy
LR_MLP, BS_MLP, EPOCHS_MLP = 0.05, 128, epochs(20)
print('MLP parameters:', sum(p.numel() for p in mlp.parameters()))

def evaluate_mlp(net, X, Y, batch_size=1000):
    total_loss, correct = 0.0, 0
    for xb, yb in batches(X, Y, batch_size):
        logits = net.forward(xb)
        total_loss += net.loss_fn.forward(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(X), correct / len(X)

hist_mlp = dict(iter_loss=[], train_loss=[], train_acc=[], val_loss=[], val_acc=[],
                steps_per_epoch=len(Xtr_flat) // BS_MLP)
t0 = time.time()
for ep in range(EPOCHS_MLP):
    losses = []
    for xb, yb in batches(Xtr_flat, Y_train, BS_MLP, shuffle=True, drop_last=True):
        losses.append(mlp.loss(xb, yb))      # forward pass
        mlp.backward()                       # hand-written backward pass
        mlp.sgd_step(LR_MLP)                 # theta <- theta - lr * dL/dtheta
    hist_mlp['iter_loss'] += torch.stack(losses).cpu().tolist()
    tl, ta = evaluate_mlp(mlp, Xtr_flat, Y_train)
    vl, va = evaluate_mlp(mlp, Xval_flat, Y_val)
    for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tl, ta, vl, va]):
        hist_mlp[k].append(v)
    print(f'epoch {ep + 1:2d} | train loss {tl:.4f} acc {100 * ta:5.2f}% | val loss {vl:.4f} acc {100 * va:5.2f}% | {time.time() - t0:5.1f}s')

best_ep = int(np.argmax(hist_mlp['val_acc']))
print(f'\nFinal validation accuracy: {100 * hist_mlp["val_acc"][-1]:.2f}%   '
      f'(best {100 * hist_mlp["val_acc"][best_ep]:.2f}% at epoch {best_ep + 1})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
ax = axes[0]
it = np.arange(1, len(hist_mlp['iter_loss']) + 1) / hist_mlp['steps_per_epoch']
ax.plot(it, hist_mlp['iter_loss'], color='tab:blue', alpha=0.25, lw=0.6, label='mini-batch loss')
plot_smoothed_loss(ax, hist_mlp, 'mini-batch loss (moving average)', k=100, color='tab:blue')
ep_axis = np.arange(1, len(hist_mlp['val_loss']) + 1)
ax.plot(ep_axis, hist_mlp['train_loss'], 'o-', color='navy', ms=3, label='training set (end of epoch)')
ax.plot(ep_axis, hist_mlp['val_loss'], 'o-', color='tab:red', ms=3, label='validation set (end of epoch)')
ax.set_xlabel('epoch'); ax.set_ylabel('cross-entropy loss'); ax.set_title('MLP trained with my SGD: loss'); ax.legend()
ax = axes[1]
ax.plot(ep_axis, 100 * np.array(hist_mlp['train_acc']), 'o-', ms=3, label='training')
ax.plot(ep_axis, 100 * np.array(hist_mlp['val_acc']), 'o-', ms=3, color='tab:red', label='validation')
ax.set_xlabel('epoch'); ax.set_ylabel('accuracy (%)'); ax.set_title('MLP: accuracy'); ax.legend()
plt.tight_layout(); plt.show()

**Interpretation (1c).** ✏️ The mini-batch loss falls quickly during the first epoch and then more slowly. It is noisy because every update uses a different random batch of 128 images, while the end-of-epoch curves are smooth. The MLP reaches **[??] %** validation accuracy after 20 epochs (best **[??] %** at epoch **[??]**). This is far above the 10% of random guessing, so the network has learned something, but it is a modest result for CIFAR-10.

Two effects limit the MLP. First, flattening discards the spatial structure of the image. Every input pixel has its own weights, so a pattern learned at one position does not transfer to another position (there is no weight sharing and no translation equivariance), and almost all of the 1.71M parameters sit in the first layer. Second, this large capacity, combined with no regularisation, leads to overfitting: the training loss keeps decreasing while the validation loss stops improving and begins to rise after about epoch **[??]**, and the gap between training accuracy (**[??] %**) and validation accuracy grows. The convolutional networks in Tasks 2–4, which share weights across positions, are far more data-efficient.

### 1(d) Per-layer gradient norms at initialisation in a 10-layer MLP: sigmoid vs ReLU

A 10-layer MLP (10 `Linear` layers, 3072–256–…–256–10) is created and, without any training, $\lVert \partial L/\partial W_\ell \rVert_F$ is computed for every layer $\ell$ with the hand-written backward pass, averaged over 10 mini-batches of 128 images. Layer 1 is next to the input and layer 10 produces the logits. Three settings are compared: **sigmoid with Xavier initialisation** (the initialisation designed for sigmoid and tanh), **sigmoid with He initialisation** (exactly the same weights as the ReLU network, so that only the activation differs) and **ReLU with He initialisation**.

In [ ]:
def layer_grad_norms(activation, init, depth=10, width=256, n_batches=10, batch_size=128):
    """Mean Frobenius norm of dL/dW for each of the `depth` Linear layers of a freshly initialised MLP."""
    seed_everything()
    net = MLP([3072] + [width] * (depth - 1) + [10], activation=activation, init=init)
    norms = torch.zeros(depth)
    for k, (xb, yb) in enumerate(batches(Xtr_flat, Y_train, batch_size, shuffle=True)):
        if k == n_batches:
            break
        net.loss(xb, yb)
        net.backward()
        norms += torch.stack([lin.dW.norm() for lin in net.linears()]).cpu()
    return (norms / n_batches).numpy()

configs_1d = {'sigmoid, Xavier init': ('sigmoid', 'xavier'),
              'sigmoid, He init (same weights as ReLU)': ('sigmoid', 'kaiming'),
              'ReLU, He init': ('relu', 'kaiming')}
grad_norms = {label: layer_grad_norms(*cfg) for label, cfg in configs_1d.items()}

layers_axis = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(8, 4.3))
for (label, g), m in zip(grad_norms.items(), ['o', 's', '^']):
    ax.semilogy(layers_axis, g, marker=m, label=label)
ax.set_xticks(layers_axis)
ax.set_xlabel('layer  (1 = next to the input, 10 = output layer)')
ax.set_ylabel(r'$\|\partial L / \partial W_\ell\|_F$  (log scale)')
ax.set_title('Gradient norm per layer at initialisation, 10-layer MLP'); ax.legend()
plt.tight_layout(); plt.show()

df_1d = pd.DataFrame(grad_norms, index=[f'layer {i}' for i in layers_axis]).T
df_1d['layer 10 / layer 1'] = df_1d['layer 10'] / df_1d['layer 1']
display(df_1d.map(lambda v: f'{v:.2e}'))

**Interpretation (1d).** ✏️ With ReLU and He initialisation the gradient norm is nearly the same in every layer. With the sigmoid it decays roughly geometrically from the output towards the input: layer 1 receives a gradient about **[??]** times smaller than layer 10 with Xavier initialisation (**[??]** times with He initialisation). This is the **vanishing gradient problem**.

The explanation is the backward recursion. With $z_\ell$ the pre-activation of layer $\ell$ and $\delta_\ell = \partial L/\partial z_\ell$,
$$\delta_\ell = \big(\delta_{\ell+1} W_{\ell+1}^\top\big) \odot \phi'(z_\ell), \qquad \frac{\partial L}{\partial W_\ell} = a_{\ell-1}^\top\,\delta_\ell .$$
Each step back multiplies the gradient variance by roughly $n\,\mathrm{Var}(W)\,\mathbb{E}[\phi'(z)^2]$, where $n=256$ is the layer width.

* **Sigmoid:** $\sigma'(z) = \sigma(z)(1-\sigma(z)) \le 1/4$, with the maximum at $z=0$, and it is smaller still when units saturate. With Xavier initialisation $n\,\mathrm{Var}(W)\approx 1$, so the gradient norm shrinks by a factor of roughly 3–4 per layer (in my run the average factor is $(\text{ratio})^{1/9} \approx$ **[??]**), which compounds to several orders of magnitude over 9 layers. He initialisation doubles $\mathrm{Var}(W)$, which slows the decay slightly but cannot compensate for $\sigma' \le 1/4$. The sigmoid output is also not zero-centred (its mean is about 0.5), which shifts the next layer's pre-activations and adds to the saturation.
* **ReLU with He initialisation:** $\phi'(z) \in \{0, 1\}$ is 1 for about half of the units, so $\mathbb{E}[\phi'^2] = 1/2$, and $n\,\mathrm{Var}(W) = n\cdot 2/n = 2$. The factor per layer is therefore $2 \cdot \tfrac12 = 1$, which is exactly the condition He initialisation was designed for, and the signal is preserved in both the forward and the backward direction. Layer 1 can differ slightly because its input is the 3072-dimensional image rather than a 256-dimensional hidden layer.

The consequence is that, in a deep sigmoid network, the early layers learn orders of magnitude more slowly than the later ones and training stalls. ReLU-type activations, variance-preserving initialisation, batch normalisation and the residual connections of Task 3(b) are all remedies for this problem.

## Task 2 — Optimizers

### 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses

Both classes follow the conventions of `torch.optim`, so that they can be compared directly with the library versions. For parameters $\theta$, gradient $g_t = \nabla L(\theta_{t-1})$, learning rate $\eta$ and L2 weight decay $\lambda$:

**SGD with momentum** (momentum $\mu$, dampening $\tau$):
$$\tilde g_t = g_t + \lambda\theta_{t-1},\qquad b_t = \mu\, b_{t-1} + (1-\tau)\,\tilde g_t\;\;(b_1 = \tilde g_1),\qquad \theta_t = \theta_{t-1} - \eta\, b_t .$$
With Nesterov momentum the update is $\theta_t = \theta_{t-1} - \eta\,(\tilde g_t + \mu b_t)$, and $\mu = 0$ gives plain SGD. This form differs from the "classical" form $v_t = \mu v_{t-1} - \eta g_t$ only when the learning rate changes during training.

**Adam** (Kingma & Ba, 2015):
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\tilde g_t,\qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\tilde g_t^{\,2},$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t},\qquad \hat v_t = \frac{v_t}{1-\beta_2^t},\qquad \theta_t = \theta_{t-1} - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}.$$
**Bias correction.** Because $m_0 = v_0 = 0$, the moving averages are biased towards zero during the first steps: $\mathbb{E}[m_t] \approx (1-\beta_1^t)\,\mathbb{E}[g]$. Dividing by $1-\beta^t$ removes this bias. Without it the first updates would have the wrong size. With $\beta_1 = 0.9$ and $\beta_2 = 0.999$, the first step would be $\frac{(1-\beta_1) g}{\sqrt{1-\beta_2}\,|g|} \approx 3.2\,\eta$ instead of $\eta$, because $v$ is much more strongly biased than $m$.

The implementation is written as $\theta \leftarrow \theta - \frac{\eta}{1-\beta_1^t}\cdot\frac{m_t}{\sqrt{v_t}/\sqrt{1-\beta_2^t} + \epsilon}$, which is algebraically identical to the formula above and is the same arrangement that PyTorch uses.

In [ ]:
class SGDMomentum(torch.optim.Optimizer):
    """SGD with optional momentum, dampening, Nesterov momentum and L2 weight decay (torch.optim.SGD conventions)."""
    def __init__(self, params, lr=1e-2, momentum=0.0, dampening=0.0, weight_decay=0.0, nesterov=False):
        if lr < 0 or momentum < 0 or weight_decay < 0:
            raise ValueError('invalid hyper-parameter')
        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError('Nesterov momentum requires momentum > 0 and zero dampening')
        super().__init__(params, dict(lr=lr, momentum=momentum, dampening=dampening,
                                      weight_decay=weight_decay, nesterov=nesterov))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, damp = group['lr'], group['momentum'], group['dampening']
            wd, nesterov = group['weight_decay'], group['nesterov']
            for p in group['params']:
                if p.grad is None:
                    continue
                d_p = p.grad
                if wd != 0:
                    d_p = d_p.add(p, alpha=wd)                       # g + lambda * theta
                if mu != 0:
                    state = self.state[p]
                    if 'momentum_buffer' not in state:               # first step: b_1 = g_1
                        buf = state['momentum_buffer'] = d_p.clone().detach()
                    else:
                        buf = state['momentum_buffer']
                        buf.mul_(mu).add_(d_p, alpha=1 - damp)       # b_t = mu b_{t-1} + (1 - tau) g_t
                    d_p = d_p.add(buf, alpha=mu) if nesterov else buf
                p.add_(d_p, alpha=-lr)                               # theta <- theta - lr * update
        return loss


class Adam(torch.optim.Optimizer):
    """Adam with bias correction and optional L2 weight decay (torch.optim.Adam conventions)."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr < 0 or eps < 0 or not (0 <= betas[0] < 1 and 0 <= betas[1] < 1):
            raise ValueError('invalid hyper-parameter')
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (beta1, beta2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)      # m
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)   # v
                state['step'] += 1
                t, m, v = state['step'], state['exp_avg'], state['exp_avg_sq']
                m.mul_(beta1).add_(g, alpha=1 - beta1)               # m_t = b1 m + (1 - b1) g
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)        # v_t = b2 v + (1 - b2) g^2
                bias_c1 = 1 - beta1 ** t                             # bias corrections
                bias_c2 = 1 - beta2 ** t
                denom = (v.sqrt() / math.sqrt(bias_c2)).add_(eps)    # sqrt(v_hat) + eps
                p.addcdiv_(m, denom, value=-lr / bias_c1)            # theta -= lr * m_hat / (sqrt(v_hat) + eps)
        return loss

**Showing that they match `torch.optim`.** Two copies of a small CNN start from identical weights. One is trained with my optimizer and the other with the PyTorch optimizer, on exactly the same sequence of 100 mini-batches of real CIFAR-10 images. After 100 steps the parameters are compared with `torch.allclose`. Several configurations are tested (plain SGD, momentum, Nesterov with weight decay, dampening, and Adam with default and non-default settings). The comparison is done in float64 so that the only possible source of difference is round-off. The column "max |Δθ| from init" shows that the parameters really moved, so a match is not trivial.

In [ ]:
def compare_with_torch(make_mine, make_ref, n_steps=100, batch_size=16, dtype=torch.float64):
    torch.manual_seed(SEED)
    model_mine = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4),
                               nn.Flatten(), nn.Linear(8 * 8 * 8, 10)).to(dtype)
    model_ref = copy.deepcopy(model_mine)                          # identical starting weights
    init = [p.detach().clone() for p in model_mine.parameters()]
    opt_mine, opt_ref = make_mine(model_mine.parameters()), make_ref(model_ref.parameters())
    X_cpu = X_train[:n_steps * batch_size].cpu().to(dtype)
    Y_cpu = Y_train[:n_steps * batch_size].cpu()
    for t in range(n_steps):
        xb, yb = X_cpu[t * batch_size:(t + 1) * batch_size], Y_cpu[t * batch_size:(t + 1) * batch_size]
        for model, opt in ((model_mine, opt_mine), (model_ref, opt_ref)):
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
    pm, pr = list(model_mine.parameters()), list(model_ref.parameters())
    return {'max |θ_mine − θ_torch|': max((a - b).abs().max().item() for a, b in zip(pm, pr)),
            'max |Δθ| from init': max((a - b).abs().max().item() for a, b in zip(pm, init)),
            'torch.allclose': all(torch.allclose(a, b) for a, b in zip(pm, pr))}

OPT_TESTS = {
    'SGD (no momentum)':
        (lambda p: SGDMomentum(p, lr=0.05), lambda p: torch.optim.SGD(p, lr=0.05)),
    'SGD, momentum 0.9':
        (lambda p: SGDMomentum(p, lr=0.05, momentum=0.9), lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    'SGD, Nesterov 0.9, weight decay 5e-4':
        (lambda p: SGDMomentum(p, lr=0.05, momentum=0.9, nesterov=True, weight_decay=5e-4),
         lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9, nesterov=True, weight_decay=5e-4)),
    'SGD, momentum 0.8, dampening 0.1':
        (lambda p: SGDMomentum(p, lr=0.05, momentum=0.8, dampening=0.1),
         lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.8, dampening=0.1)),
    'Adam (lr 1e-3, default betas)':
        (lambda p: Adam(p, lr=1e-3), lambda p: torch.optim.Adam(p, lr=1e-3)),
    'Adam (lr 3e-3, betas (0.8, 0.99), wd 1e-4)':
        (lambda p: Adam(p, lr=3e-3, betas=(0.8, 0.99), weight_decay=1e-4),
         lambda p: torch.optim.Adam(p, lr=3e-3, betas=(0.8, 0.99), weight_decay=1e-4)),
}
df_opt = pd.DataFrame({name: compare_with_torch(*makers) for name, makers in OPT_TESTS.items()}).T
for col in ['max |θ_mine − θ_torch|', 'max |Δθ| from init']:
    df_opt[col] = df_opt[col].map(lambda v: f'{v:.2e}')
display(df_opt)

**Interpretation (2a).** ✏️ In every configuration the parameters after 100 steps agree with those of `torch.optim.SGD` / `torch.optim.Adam` to within **[??]** (of order $10^{-15}$ or smaller), and `torch.allclose` is `True`, even though the parameters themselves moved by up to **[??]**. The remaining differences are round-off: PyTorch, for example, updates $m$ with `lerp_` while my code uses `mul_` and `add_`, which are mathematically equal but round differently. Matching over many steps also confirms that the state (momentum buffers, $m$, $v$ and the step counter used in the bias correction) is stored and updated correctly, because an error there would accumulate. In the rest of the notebook my `SGDMomentum` and `Adam` are used in Task 2. Tasks 3 and 4 use the equivalent `torch.optim` classes because their vectorised implementations are faster.

### 2(b) Trajectories on $f(x, y) = \tfrac12\,(x^2 + 50\,y^2)$

The gradient is $\nabla f = (x,\; 50y)$ and the Hessian is $\mathrm{diag}(1, 50)$, so the curvature along $y$ is 50 times larger than along $x$ (condition number $\kappa = 50$). The level sets are long, narrow ellipses. All three methods start at $(-10, 1)$ and take 100 steps using my optimizer classes from 2(a). The gradient here is exact, so "SGD" is simply gradient descent. Plain SGD and SGD with momentum use the same learning rate $\eta = 0.035$, close to the stability limit $2/50 = 0.04$ set by the steep direction, and momentum uses $\mu=0.9$. Adam uses $\eta = 0.3$, because its step size is set per coordinate (see below), so its learning rate is on a different scale. Note that the two axes of the contour plot have different scales.

In [ ]:
def f_quad(w):
    return 0.5 * (w[0] ** 2 + 50 * w[1] ** 2)

def trajectory(make_opt, n_steps=100, start=(-10.0, 1.0)):
    w = torch.tensor(start, dtype=torch.float64, requires_grad=True)
    opt = make_opt([w])
    path = [w.detach().clone()]
    for _ in range(n_steps):
        opt.zero_grad()
        f_quad(w).backward()              # gradient (x, 50 y)
        opt.step()
        path.append(w.detach().clone())
    return torch.stack(path).numpy()

TRAJ = {'SGD (η = 0.035)':                   lambda p: SGDMomentum(p, lr=0.035),
        'SGD + momentum (η = 0.035, μ = 0.9)': lambda p: SGDMomentum(p, lr=0.035, momentum=0.9),
        'Adam (η = 0.3)':                    lambda p: Adam(p, lr=0.3)}
paths = {name: trajectory(make) for name, make in TRAJ.items()}

xs, ys = np.meshgrid(np.linspace(-11, 11, 500), np.linspace(-2.2, 2.2, 500))
zs = 0.5 * (xs ** 2 + 50 * ys ** 2)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.3, 1]})
ax = axes[0]
cs = ax.contour(xs, ys, zs, levels=np.logspace(-2, 2.2, 16), cmap='Greys', linewidths=0.7)
for (name, p), c in zip(paths.items(), ['tab:red', 'tab:blue', 'tab:green']):
    ax.plot(p[:, 0], p[:, 1], '-o', ms=2.5, lw=1.1, color=c, label=name)
ax.plot(-10, 1, 'ks', ms=7, label='start (−10, 1)')
ax.plot(0, 0, 'k*', ms=13, label='minimum (0, 0)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_xlim(-11, 11); ax.set_ylim(-2.2, 2.2)
ax.set_title(r'Trajectories on $f(x,y)=\frac{1}{2}(x^2+50y^2)$ (100 steps; axes not to scale)')
ax.legend(loc='lower right', fontsize=8)
ax = axes[1]
for (name, p), c in zip(paths.items(), ['tab:red', 'tab:blue', 'tab:green']):
    ax.semilogy(0.5 * (p[:, 0] ** 2 + 50 * p[:, 1] ** 2), color=c, label=name)
ax.set_xlabel('iteration'); ax.set_ylabel('f (log scale)'); ax.set_title('Objective value'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for name, p in paths.items():
    k_x = int(np.argmax(p[:, 0]))
    print(f'{name:38s} final (x, y) = ({p[-1, 0]: .4f}, {p[-1, 1]: .2e})   f = {f_quad(torch.tensor(p[-1])).item():.2e}'
          f'   largest x reached = {p[k_x, 0]:.2f} (iteration {k_x})')

**Interpretation (2b).** Because $f$ is a sum of independent quadratics, plain gradient descent acts on each coordinate separately: $x_{t+1} = (1-\eta)\,x_t$ and $y_{t+1} = (1-50\eta)\,y_t$.

* **SGD.** With $\eta = 0.035$ the factor along $y$ is $1 - 1.75 = -0.75$, so $y$ changes sign at every step. The red path zig-zags across the narrow valley while the oscillation slowly dies out. Along the flat $x$ direction the factor is $0.965$, so progress is slow: after 100 steps $x \approx -10\cdot 0.965^{100} \approx -0.28$ and $f \approx 4\times10^{-2}$. The learning rate cannot be increased to speed this up, because $\eta > 2/50$ would make the $y$ iteration diverge. The steepest direction limits the step size, while the flattest direction determines how fast the method converges. This is the classic ill-conditioning problem.
* **SGD with momentum.** The velocity accumulates along $x$, where successive gradients have the same sign, and the effective step there grows towards $\eta/(1-\mu) = 10\eta$. The blue path therefore covers the flat direction much faster and ends at $f \approx 1.5\times 10^{-3}$, about 25 times lower than SGD. The price is oscillation. With $\mu = 0.9$ the heavy-ball iteration is under-damped in *both* directions: its error contracts by only $\sqrt{\mu} \approx 0.95$ per step (against $0.75$ for plain SGD along $y$), so the path keeps bouncing across the valley in $y$ for longer than plain SGD does. Along $x$ it overshoots the minimum, reaching $x \approx +4.1$ at iteration 16, and then spirals in. The jagged, non-monotone objective curve on the right shows both effects. A smaller momentum ($\mu \approx 0.57$ is optimal for $\kappa = 50$) would damp the oscillations while keeping most of the speed-up.
* **Adam.** Adam divides each coordinate's step by a running estimate of that coordinate's gradient magnitude, so the first steps are about $\eta\cdot\mathrm{sign}(g)$ in each coordinate, whatever the curvature. The green path therefore starts out at about 45° in parameter space, ignoring the 50:1 difference in curvature. Adam is invariant to a rescaling of each coordinate, which is exactly the mismatch that hurts SGD here. Once $y$ is near zero, $x$ moves at almost a constant speed of about $\eta$ per step, and Adam is close to the minimum after about 40 steps. Near the minimum the step does not shrink in proportion to the gradient, because $\hat m/\sqrt{\hat v}$ stays near $\pm 1$ until $v$ adapts. Adam therefore circles the optimum at a scale set by $\eta$ and, with a constant learning rate, does not converge exactly: it ends at $f \approx 1.4\times10^{-3}$, similar to momentum. A decaying learning rate would remove this residual oscillation.

In short, momentum counters ill-conditioning by building up speed along the consistently-signed flat direction, and Adam counters it by rescaling each coordinate. Both end about 25–30 times lower in $f$ than plain SGD after 100 steps; plain SGD would need about 150 steps to reach the same value.

### 2(c) A small CNN trained with each optimizer at three learning rates

**Network** (`SmallCNN`): three stages of [conv3×3 → ReLU → conv3×3 → ReLU → max-pool 2×2] with 32, 64 and 128 channels, followed by a linear classifier on the flattened 128×4×4 feature map (about 0.31M parameters). The conv layers use He initialisation and the classifier keeps PyTorch's default initialisation, which gives small initial logits. Batch normalisation is deliberately **left out** here, because it makes networks much less sensitive to the learning rate and would hide the differences between the optimizers.

**Protocol.** 10 epochs on the 45k training images, batch size 128, a constant learning rate and no augmentation or regularisation. Every run starts from the same initial weights and sees the same sequence of batches (the seed is reset before each run), so the optimizer and the learning rate are the only differences. The learning rates are spaced by factors of 10: $\{10^{-3}, 10^{-2}, 10^{-1}\}$ for SGD and SGD with momentum ($\mu = 0.9$), and $\{10^{-4}, 10^{-3}, 10^{-2}\}$ for Adam, whose usual range is about ten times lower. The validation accuracy after the last epoch is reported.

In [ ]:
class SmallCNN(nn.Module):
    """[conv-(BN)-ReLU-conv-(BN)-ReLU-maxpool-(dropout)] x 3 with width, 2*width, 4*width channels, then a linear layer."""
    def __init__(self, width=32, batch_norm=False, dropout=0.0, num_classes=10):
        super().__init__()
        chans = [3, width, 2 * width, 4 * width]
        layers = []
        for i in range(3):
            for j in range(2):
                c_in = chans[i] if j == 0 else chans[i + 1]
                layers.append(nn.Conv2d(c_in, chans[i + 1], 3, padding=1, bias=not batch_norm))
                if batch_norm:
                    layers.append(nn.BatchNorm2d(chans[i + 1]))
                layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(2))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(chans[3] * 4 * 4, num_classes)
        for m in self.modules():   # He init for the convs; the linear classifier keeps PyTorch's default (smaller) init
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

print('SmallCNN parameters (no BN):', f'{count_params(SmallCNN()):,}')

In [ ]:
OPT_GRID = {
    'SGD':          ([1e-3, 1e-2, 1e-1], lambda params, lr: SGDMomentum(params, lr=lr)),
    'SGD+momentum': ([1e-3, 1e-2, 1e-1], lambda params, lr: SGDMomentum(params, lr=lr, momentum=0.9)),
    'Adam':         ([1e-4, 1e-3, 1e-2], lambda params, lr: Adam(params, lr=lr)),
}
EPOCHS_2C, BS_2C = epochs(10), 128

results_2c = {}
for opt_name, (lrs, make_opt) in OPT_GRID.items():
    for lr in lrs:
        run_name = f'{opt_name} lr={lr:g}'
        def run(make_opt=make_opt, lr=lr, run_name=run_name):
            seed_everything()                     # same initial weights and batch order for every run
            model = SmallCNN(batch_norm=False).to(device)
            return train(model, make_opt(model.parameters(), lr), EPOCHS_2C, X_train, Y_train, X_val, Y_val,
                         batch_size=BS_2C, name=run_name)
        results_2c[(opt_name, lr)] = cached('task2c_' + run_name, run)

In [ ]:
# ---- table: final validation accuracy for every (optimizer, learning rate) ----
rows = []
for (opt_name, lr), h in results_2c.items():
    rows.append({'optimizer': opt_name, 'learning rate': lr, 'final val acc (%)': 100 * h['val_acc'][-1],
                 'best val acc (%)': 100 * max(h['val_acc']), 'final train loss': h['train_loss'][-1],
                 'diverged': h['diverged']})
df_2c = pd.DataFrame(rows)
display(df_2c.round(3).assign(**{'learning rate': df_2c['learning rate'].map(lambda v: f'{v:g}')}))
best_lr = {o: float(df_2c[df_2c.optimizer == o].sort_values('final val acc (%)').iloc[-1]['learning rate']) for o in OPT_GRID}
print('best learning rate per optimizer:', best_lr)

colors_2c = {'SGD': 'tab:red', 'SGD+momentum': 'tab:blue', 'Adam': 'tab:green'}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ax = axes[0]
for o, (lrs, _) in OPT_GRID.items():
    accs = [100 * results_2c[(o, lr)]['val_acc'][-1] for lr in lrs]
    ax.semilogx(lrs, accs, 'o-', color=colors_2c[o], label=o, ms=6)
    for lr, a in zip(lrs, accs):
        if results_2c[(o, lr)]['diverged']:
            ax.annotate('diverged', (lr, a), textcoords='offset points', xytext=(-20, 8), fontsize=8, color=colors_2c[o])
ax.axhline(10, color='grey', ls=':', lw=1, label='chance (10%)')
ax.set_xlabel('learning rate (log scale)'); ax.set_ylabel('validation accuracy after 10 epochs (%)')
ax.set_title('Learning-rate sensitivity'); ax.legend()
ax = axes[1]
for o in OPT_GRID:
    plot_smoothed_loss(ax, results_2c[(o, best_lr[o])], f'{o} (lr={best_lr[o]:g})', color=colors_2c[o])
ax.set_xlabel('epoch'); ax.set_ylabel('training loss (moving average)'); ax.set_title('Training loss at the best learning rate'); ax.legend()
ax = axes[2]
for o in OPT_GRID:
    h = results_2c[(o, best_lr[o])]
    ax.plot(np.arange(1, len(h['val_acc']) + 1), 100 * np.array(h['val_acc']), 'o-', ms=3, color=colors_2c[o],
            label=f'{o} (lr={best_lr[o]:g})')
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy (%)'); ax.set_title('Validation accuracy at the best learning rate'); ax.legend()
plt.tight_layout(); plt.show()

**Interpretation (2c).** ✏️ *(Check every claim below against your table and plots and correct it where necessary.)*

* **Plain SGD** is the most sensitive at the low end. At $\eta = 10^{-3}$ the network barely trains in 10 epochs (**[??] %**) because each step is tiny. Accuracy rises steadily with $\eta$, and the best value is $\eta=$ **[??]** (**[??] %**). Its useful range lies at the high end of the grid, so a larger learning rate might be better still.
* **SGD with momentum** behaves like SGD with a learning rate about $1/(1-\mu) = 10$ times larger, because the velocity sums roughly 10 recent gradients. Its whole curve is therefore shifted about one decade to the left of the SGD curve. It is best at $\eta=$ **[??]** (**[??] %**), and at $\eta = 0.1$ the effective step of about 1 is too large for a network without batch normalisation: the run **[diverges / is unstable]** and ends near chance level. Momentum gives the best accuracy here, but its learning rate has to be chosen within about one decade.
* **Adam** reaches its best result at $\eta = 10^{-3}$ (**[??] %**), its usual default, and still performs reasonably at $10^{-4}$, because the per-parameter normalisation makes the step size roughly independent of the scale of the gradients. At $10^{-2}$ the normalised steps (about $\eta$ per weight, whatever the gradient) are too large and the accuracy drops to **[??] %**.
* **Training loss at the best rates.** **[Adam / momentum]** lowers the training loss fastest in the first epoch, and plain SGD is the slowest even at its best rate. By the end of the 10 epochs **[momentum and Adam are close / …]**.

**Sensitivity.** Every optimizer works well only within about one decade of learning rates, but the location of that window differs. It is set by the scale of the raw gradient for SGD, by the gradient scale times $1/(1-\mu)$ for momentum, and by the target step size per parameter for Adam. Adam's window was the easiest to hit with its default value. SGD-type methods depend more on tuning, but with a well-chosen learning rate (and, in practice, a schedule) momentum is usually at least as accurate. Ten epochs at a constant learning rate are short, so the slower configurations have not converged. The comparison therefore measures speed of learning as much as final quality.

## Task 3 — Regularisation and residual connections

### 3(a) Regularisation on 5,000 training images

The `SmallCNN` from 2(c), now **with batch normalisation**, is trained on only 5,000 training images (the first 5k of the already random 45k split), which makes overfitting severe. The full 5k validation set is used for evaluation. Seven settings are compared:

| setting | what it does |
|---|---|
| baseline | no regularisation |
| weight decay | L2 penalty $\lambda = 5\times10^{-4}$, i.e. $\lambda\theta$ is added to every gradient |
| dropout | `nn.Dropout(p=0.3)` after each of the three pooling stages (so also before the classifier) |
| flip + crop | random horizontal flip, then a random 32×32 crop from the image zero-padded by 4 pixels |
| cutout | a random 16×16 square is set to zero (the mean colour) in every image (DeVries & Taylor, 2017) |
| mixup | train on $\lambda x_i + (1-\lambda)x_j$ with loss $\lambda\,\ell(y_i) + (1-\lambda)\,\ell(y_j)$, where $\lambda\sim\mathrm{Beta}(0.4, 0.4)$ (Zhang et al., 2018) |
| all combined | all of the above together |

Cutout and mixup are implemented by hand in the utilities section (`cutout`, `mixup`). **Training:** SGD with Nesterov momentum 0.9, learning rate 0.05 with a 3-epoch linear warm-up and cosine decay, batch size 128, 60 epochs, identical initialisation for every setting. **Train accuracy** is measured on the *clean* 5k training images in evaluation mode at the end of each epoch. Measuring it on augmented or mixed batches would not be comparable across settings, and the gap must compare like with like.

First, a visual check that the augmentations do what they should:

In [ ]:
seed_everything(0)
sample, labels = X_train[:8], Y_train[:8]
x_mix, y_a, y_b, lam = mixup(sample, labels, alpha=0.4)
show_rows([('original', sample, labels),
           ('flip + crop', random_crop(random_flip(sample)), labels),
           ('cutout 16×16', cutout(sample, 16), labels),
           (f'mixup λ={lam:.2f}', x_mix, y_a)], n=8)

In [ ]:
N_SMALL = 5000
X_small, Y_small = X_train[:N_SMALL], Y_train[:N_SMALL]     # the split is already random -> random 5k subset
EPOCHS_3A, BS_3A, LR_3A = epochs(60), 128, 0.05
WD, P_DROP, CUTOUT_SIZE, MIXUP_ALPHA = 5e-4, 0.3, 16, 0.4

REG_CONFIGS = {
    'baseline':     dict(),
    'weight decay': dict(weight_decay=WD),
    'dropout':      dict(dropout=P_DROP),
    'flip + crop':  dict(flip_crop=True),
    'cutout':       dict(cutout=CUTOUT_SIZE),
    'mixup':        dict(mixup=MIXUP_ALPHA),
    'all combined': dict(weight_decay=WD, dropout=P_DROP, flip_crop=True, cutout=CUTOUT_SIZE, mixup=MIXUP_ALPHA),
}
print('SmallCNN with BN:', f'{count_params(SmallCNN(batch_norm=True)):,}', 'parameters')

results_3a = {}
for name, cfg in REG_CONFIGS.items():
    def run(cfg=cfg, name=name):
        seed_everything()
        model = SmallCNN(batch_norm=True, dropout=cfg.get('dropout', 0.0)).to(device)
        opt = torch.optim.SGD(model.parameters(), lr=LR_3A, momentum=0.9, nesterov=True,
                              weight_decay=cfg.get('weight_decay', 0.0))
        return train(model, opt, EPOCHS_3A, X_small, Y_small, X_val, Y_val, batch_size=BS_3A,
                     augment=make_augment(cfg.get('flip_crop', False), cfg.get('cutout', 0)),
                     mixup_alpha=cfg.get('mixup', 0.0), lr_schedule=cosine_schedule(warmup_frac=0.05),
                     eval_train=(X_small, Y_small), name=name)
    results_3a[name] = cached('task3a_' + name, run)

In [ ]:
rows = []
for name, h in results_3a.items():
    tr, va = 100 * h['train_acc'][-1], 100 * h['val_acc'][-1]
    rows.append({'setting': name, 'train acc (%)': tr, 'val acc (%)': va, 'gap (pp)': tr - va,
                 'best val acc (%)': 100 * max(h['val_acc']), 'final val loss': h['val_loss'][-1]})
df_3a = pd.DataFrame(rows).set_index('setting')
display(df_3a.round(2))

fig, axes = plt.subplots(2, 4, figsize=(17, 7.5), sharey=True)
for ax, (name, h) in zip(axes.flat, results_3a.items()):
    ep = np.arange(1, len(h['val_acc']) + 1)
    ax.plot(ep, 100 * np.array(h['train_acc']), label='train (clean images)')
    ax.plot(ep, 100 * np.array(h['val_acc']), label='validation')
    ax.fill_between(ep, 100 * np.array(h['val_acc']), 100 * np.array(h['train_acc']), alpha=0.12, color='grey')
    ax.set_title(f'{name}  (gap {df_3a.loc[name, "gap (pp)"]:.1f} pp)', fontsize=10)
    ax.set_xlabel('epoch')
for ax in axes[:, 0]:
    ax.set_ylabel('accuracy (%)')
axes[0, 0].legend(fontsize=8)
ax = axes.flat[-1]
pos = np.arange(len(df_3a))
ax.bar(pos - 0.2, df_3a['train acc (%)'], 0.4, label='train')
ax.bar(pos + 0.2, df_3a['val acc (%)'], 0.4, label='validation')
ax.set_xticks(pos); ax.set_xticklabels(df_3a.index, rotation=45, ha='right', fontsize=8)
ax.set_title('final accuracies', fontsize=10); ax.legend(fontsize=8)
plt.suptitle('Task 3(a): SmallCNN with BN on 5k training images', y=1.0)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
for name, h in results_3a.items():
    ep = np.arange(1, len(h['val_acc']) + 1)
    lw = 2.5 if name in ('baseline', 'all combined') else 1.2
    axes[0].plot(ep, h['val_loss'], lw=lw, label=name)
    axes[1].plot(ep, 100 * np.array(h['val_acc']), lw=lw, label=name)
axes[0].set_ylabel('validation loss'); axes[1].set_ylabel('validation accuracy (%)')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend(fontsize=8)
axes[0].set_ylim(top=min(4.0, axes[0].get_ylim()[1]))
plt.tight_layout(); plt.show()

**Interpretation (3a).** ✏️ *(Correct the ranking and the numbers to match your table.)*

* **Baseline.** With 0.3M parameters and only 5k images the network memorises the training set: training accuracy reaches **[??] %** while validation accuracy is only **[??] %**, a gap of **[??]** percentage points. The validation loss reaches a minimum early and then **rises**, because the network becomes increasingly over-confident on validation images it gets wrong. This is classic overfitting.
* **Weight decay** changes little (**[??] %**, gap **[??]** pp). With batch normalisation, the output of a conv layer does not change when its weights are scaled, so L2 decay mostly increases the effective learning rate instead of limiting capacity, and $5\times10^{-4}$ over about 2,400 steps is a mild penalty.
* **Dropout** reduces the gap somewhat (**[??]** pp) by stopping units from co-adapting, but it does not add any new information about the data.
* **Flip + crop** is the most effective single regulariser (**[??] %** validation accuracy). It encodes invariances that really hold for natural images: a mirrored or slightly shifted cat is still a cat. This multiplies the effective size of the training set, and the train accuracy (on clean images) **stays high while the validation accuracy rises**, so the gap closes from the validation side.
* **Cutout** forces the network to use many parts of the object instead of a single discriminative patch, which improves validation accuracy to **[??] %**.
* **Mixup** trains on convex combinations of images and labels. This encourages linear behaviour between training examples and prevents over-confident predictions, which shows up clearly as a much lower validation **loss** (**[??]** vs **[??]** for the baseline), even where the accuracy gain is smaller.
* **All combined** gives the smallest gap (**[??]** pp) and **[the highest / a lower]** validation accuracy (**[??] %**). The regularisers are not additive: together they make the task so hard that the network **[still under-fits after 60 epochs / …]**, so its training accuracy falls well below 100%. A larger model or a longer schedule would benefit more from this combination.

Overall, the techniques that **add information about the data distribution** (augmentation) helped much more than those that only **restrict the model** (weight decay, dropout), which is the expected behaviour in the small-data regime.

### 3(b) Plain vs residual networks of depth $6n+2$ (8, 20, 56)

**Architecture** (He et al., 2016, Section 4.2): a 3×3 conv with 16 channels, then three stages of $n$ *basic blocks* with 16, 32 and 64 channels on 32×32, 16×16 and 8×8 feature maps, then global average pooling and a linear layer, for $2 + 6n$ weighted layers. A basic block is conv3×3 → BN → ReLU → conv3×3 → BN, then (residual only) **+ shortcut**, then ReLU. The first block of stages 2 and 3 halves the resolution with stride 2. All conv weights use Kaiming (He) normal initialisation, BN weights and biases start at 1 and 0, and the final linear layer keeps PyTorch's default initialisation.

**The only difference between the two families is the shortcut.** The residual network uses parameter-free identity shortcuts. Where the shape changes, it uses He et al.'s *option A*: the input is subsampled with stride 2 and the extra channels are padded with zeros. As a result the plain and residual networks of the same depth have *exactly* the same parameters (checked below). With $n = 1, 3, 9$ the depths are 8, 20 and 56.

**Training:** 20 epochs on the 45k training images with flip + crop, SGD with momentum 0.9, learning rate 0.1 (with a 1-epoch warm-up and cosine decay), weight decay $10^{-4}$, batch size 128. These are the settings of the original paper, apart from the shortened schedule.

In [ ]:
class BasicBlock(nn.Module):
    """conv3x3-BN-ReLU-conv3x3-BN (+ identity shortcut) -ReLU.  With shortcut=False the block is 'plain'."""
    def __init__(self, c_in, c_out, stride=1, shortcut=True):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut, self.stride, self.extra = shortcut, stride, c_out - c_in

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.shortcut:
            sc = x
            if self.stride != 1 or self.extra != 0:          # option A: subsample + zero-pad channels (no parameters)
                sc = x[:, :, ::self.stride, ::self.stride]
                sc = F.pad(sc, (0, 0, 0, 0, self.extra // 2, self.extra - self.extra // 2))
            out = out + sc
        return F.relu(out)


class CifarResNet(nn.Module):
    """CIFAR-style network: conv3x3 stem, three stages of BasicBlocks, global average pooling, linear layer."""
    def __init__(self, blocks=(3, 3, 3), widths=(16, 32, 64), shortcut=True, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, c_in = [], widths[0]
        for stage, (n, w) in enumerate(zip(blocks, widths)):
            for b in range(n):
                stride = 2 if (stage > 0 and b == 0) else 1      # downsample at the start of stages 2 and 3
                layers.append(BasicBlock(c_in, w, stride, shortcut))
                c_in = w
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(c_in, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        # the final linear layer keeps PyTorch's default initialisation (small initial logits)

    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))   # global average pooling


def cifar_net(depth, shortcut):
    assert (depth - 2) % 6 == 0, 'depth must be 6n + 2'
    n = (depth - 2) // 6
    return CifarResNet((n, n, n), (16, 32, 64), shortcut)

DEPTHS = [8, 20, 56]
df_params = pd.DataFrame([{'depth': d, 'n (blocks per stage)': (d - 2) // 6,
                           'plain: parameters': count_params(cifar_net(d, False)),
                           'residual: parameters': count_params(cifar_net(d, True))} for d in DEPTHS]).set_index('depth')
display(df_params)

In [ ]:
EPOCHS_3B = epochs(20)
results_3b = {}
for depth in DEPTHS:
    for shortcut in [False, True]:
        run_name = f'{"ResNet" if shortcut else "plain"}-{depth}'
        def run(depth=depth, shortcut=shortcut, run_name=run_name):
            seed_everything()
            model = cifar_net(depth, shortcut).to(device)
            opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
            return train(model, opt, EPOCHS_3B, X_train, Y_train, X_val, Y_val, batch_size=128,
                         augment=make_augment(flip_crop=True), lr_schedule=cosine_schedule(warmup_frac=0.05),
                         name=run_name)
        results_3b[(depth, shortcut)] = cached('task3b_' + run_name, run)

In [ ]:
colors_3b = {8: 'tab:green', 20: 'tab:blue', 56: 'tab:red'}
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for (depth, shortcut), h in results_3b.items():
    ep = np.arange(1, len(h['train_loss']) + 1)
    style = dict(ls='-' if shortcut else '--', marker='o' if shortcut else 's', ms=3, color=colors_3b[depth],
                 label=f'{"ResNet" if shortcut else "plain"}-{depth}')
    axes[0].plot(ep, h['train_loss'], **style)
    axes[1].plot(ep, 100 * np.array(h['val_acc']), **style)
    if depth == 56:
        plot_smoothed_loss(axes[2], h, style['label'], k=100, color=colors_3b[depth], ls=style['ls'])
axes[0].set_ylabel('training loss (mean over the epoch)'); axes[0].set_title('Training loss vs epoch')
axes[1].set_ylabel('validation accuracy (%)'); axes[1].set_title('Validation accuracy vs epoch')
axes[2].set_ylabel('training loss (moving average)'); axes[2].set_title('Depth 56: per-iteration training loss')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

rows = []
for (depth, shortcut), h in results_3b.items():
    rows.append({'network': f'{"ResNet" if shortcut else "plain"}-{depth}', 'parameters': df_params.loc[depth, 'plain: parameters'],
                 'train loss, epoch 1': h['train_loss'][0], 'final train loss': h['train_loss'][-1],
                 'final val acc (%)': 100 * h['val_acc'][-1], 'time (s)': h['time']})
display(pd.DataFrame(rows).set_index('network').round(3))

**Interpretation (3b).** ✏️

* **Plain networks get worse with depth.** Plain-20 ends with a **higher** training loss than plain-8 **[check]**, and plain-56 is clearly the worst: its loss falls slowly from the start and ends at **[??]**, compared with **[??]** for plain-20. This is the *degradation problem* of He et al. It is **not overfitting**, because it is the *training* loss that is worse (validation accuracy follows the same order: **[??] %** for plain-56 vs **[??] %** for plain-20). In principle a 56-layer network can represent everything a 20-layer one can (the extra layers could implement the identity), so the problem is that SGD cannot *find* such a solution. Batch normalisation keeps the forward signals and the gradients at a reasonable scale at initialisation, so this is not simply vanishing gradients as in Task 1(d). Deep plain stacks are nevertheless badly conditioned, and the gradient that reaches the early layers becomes increasingly unstructured with depth ("shattered gradients"), which slows optimisation.
* **Residual networks get better with depth.** With identity shortcuts the order reverses: ResNet-56 reaches the **lowest** training loss (**[??]**), then ResNet-20, then ResNet-8, and the validation accuracy rises with depth too (**[??] %** for ResNet-56). Each block computes $x_{\ell+1} = \mathrm{ReLU}(x_\ell + F(x_\ell))$, so it only has to learn a *residual* correction to the identity, and the identity (the "do nothing" solution) is available by driving $F$ towards 0. In the backward pass, $\partial x_L/\partial x_\ell$ contains a direct identity term in addition to the products of the weight Jacobians, so gradients reach the early layers along short paths and the network behaves at the start of training like an ensemble of shallow networks.
* **At depth 8** the two curves almost coincide, because there are only three blocks and optimisation is not yet a problem. The gap between plain and residual grows with depth and is largest at 56 layers, even though the two networks have identical parameters. The shortcut changes only the *optimisation landscape*, not the capacity.

## Task 4 — The best CNN with at most 500k parameters and 20 training epochs (optional)

**Design choices and their rationale**

* **Residual network with BN** (the `CifarResNet` of 3(b)) with option-A identity shortcuts, because Task 3(b) showed that they make optimisation easier, and global average pooling, which keeps the classifier tiny.
* **Width over depth.** With only 20 epochs, wide and moderately deep networks learn faster per epoch than very deep and narrow ones. Three architectures under the budget are compared on the validation set: a wide 8-layer network (40-80-160 channels), a 12-layer network (32-64-128) and a narrower 14-layer network (24-48-96).
* **Augmentation:** flip + crop + an 8×8 cutout. A smaller patch than in 3(a), because a short schedule cannot absorb strong regularisation.
* **Optimisation:** SGD with Nesterov momentum 0.9, batch size 256, peak learning rate 0.2 (0.1 per 128 images), a linear warm-up over the first 15% of the steps and then cosine decay to 0. This is a "one-cycle"-like schedule that is well suited to a fixed epoch budget. Weight decay $5\times10^{-4}$ is applied to conv and linear weights only, not to BN parameters or biases.
* **Label smoothing 0.1**, which discourages over-confident logits.

**Procedure.** (1) Train the three candidate architectures with this recipe and choose the one with the best **validation** accuracy. (2) Ablate four design choices on the chosen architecture, one at a time: the shortcut, the augmentation, the weight decay and the label smoothing. (3) Evaluate the chosen model **once** on the test set. Every run uses at most 20 epochs.

In [ ]:
T4_EPOCHS, T4_BS = epochs(20), 256
T4_CANDIDATES = {
    'ResNet-8 (40-80-160)':  dict(blocks=(1, 1, 1), widths=(40, 80, 160)),
    'ResNet-12 (32-64-128)': dict(blocks=(2, 2, 1), widths=(32, 64, 128)),
    'ResNet-14 (24-48-96)':  dict(blocks=(2, 2, 2), widths=(24, 48, 96)),
}
BASE_RECIPE = dict(shortcut=True, flip_crop=True, cutout=8, weight_decay=5e-4, label_smoothing=0.1, lr=0.2)

def sgd_no_decay_on_bn(model, lr, weight_decay):
    """Nesterov SGD; weight decay only on conv/linear weights (tensors with >1 dimension), not on BN or biases."""
    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]
    return torch.optim.SGD([{'params': decay, 'weight_decay': weight_decay},
                            {'params': no_decay, 'weight_decay': 0.0}], lr=lr, momentum=0.9, nesterov=True)

def build_t4(recipe):
    return CifarResNet(recipe['blocks'], recipe['widths'], recipe['shortcut'])

def run_t4(name, recipe):
    def _run():
        seed_everything()
        model = build_t4(recipe).to(device)
        n_params = count_params(model)
        assert n_params <= 500_000, f'{name}: {n_params} parameters exceed the 500k budget'
        opt = sgd_no_decay_on_bn(model, recipe['lr'], recipe['weight_decay'])
        hist = train(model, opt, T4_EPOCHS, X_train, Y_train, X_val, Y_val, batch_size=T4_BS,
                     augment=make_augment(recipe['flip_crop'], recipe['cutout']),
                     label_smoothing=recipe['label_smoothing'], lr_schedule=cosine_schedule(warmup_frac=0.15), name=name)
        return dict(hist=hist, params=n_params, recipe=recipe,
                    state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()})
    return cached('task4_' + name, _run)

display(pd.DataFrame({name: {'blocks per stage': str(c['blocks']), 'channels': str(c['widths']),
                             'parameters': count_params(build_t4({**BASE_RECIPE, **c}))}
                      for name, c in T4_CANDIDATES.items()}).T)

In [ ]:
# ---- step 1: choose the architecture on the VALIDATION set ----
cand_results = {name: run_t4(name, {**BASE_RECIPE, **c}) for name, c in T4_CANDIDATES.items()}

df_cand = pd.DataFrame({name: {'parameters': r['params'], 'final val acc (%)': 100 * r['hist']['val_acc'][-1],
                               'final train loss': r['hist']['train_loss'][-1], 'time (s)': r['hist']['time']}
                        for name, r in cand_results.items()}).T
display(df_cand.round(3))
best_name = max(cand_results, key=lambda k: cand_results[k]['hist']['val_acc'][-1])
best_recipe = cand_results[best_name]['recipe']
print('chosen architecture:', best_name)

fig, ax = plt.subplots(figsize=(8, 4))
for name, r in cand_results.items():
    ax.plot(np.arange(1, len(r['hist']['val_acc']) + 1), 100 * np.array(r['hist']['val_acc']), 'o-', ms=3,
            label=f'{name}, {r["params"] / 1e3:.0f}k params')
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy (%)'); ax.set_title('Task 4: candidate architectures'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---- step 2: ablations on the chosen architecture (one change at a time) ----
ABLATIONS = {
    'no shortcut (plain network)': dict(shortcut=False),
    'no augmentation':             dict(flip_crop=False, cutout=0),
    'no weight decay':             dict(weight_decay=0.0),
    'no label smoothing':          dict(label_smoothing=0.0),
}
abl_results = {'full model': cand_results[best_name]}
for abl_name, change in ABLATIONS.items():
    abl_results[abl_name] = run_t4(f'{best_name} | {abl_name}', {**best_recipe, **change})

full_acc = 100 * abl_results['full model']['hist']['val_acc'][-1]
df_abl = pd.DataFrame({name: {'parameters': r['params'], 'val acc (%)': 100 * r['hist']['val_acc'][-1],
                              'Δ vs full (pp)': 100 * r['hist']['val_acc'][-1] - full_acc,
                              'final train loss': r['hist']['train_loss'][-1]}
                       for name, r in abl_results.items()}).T
display(df_abl.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
for name, r in abl_results.items():
    axes[0].plot(np.arange(1, len(r['hist']['val_acc']) + 1), 100 * np.array(r['hist']['val_acc']),
                 lw=2.5 if name == 'full model' else 1.2, label=name)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('validation accuracy (%)'); axes[0].legend(fontsize=8)
axes[0].set_title(f'Ablations of {best_name}')
axes[1].barh(df_abl.index[::-1], df_abl['val acc (%)'][::-1], color=['tab:gray'] * (len(df_abl) - 1) + ['tab:green'])
for i, v in enumerate(df_abl['val acc (%)'][::-1]):
    axes[1].text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=8)
axes[1].set_xlim(df_abl['val acc (%)'].min() - 3, df_abl['val acc (%)'].max() + 1.5)
axes[1].set_xlabel('final validation accuracy (%)'); axes[1].set_title('Ablation summary')
plt.tight_layout(); plt.show()

**Interpretation (ablations).** ✏️ The chosen architecture is **[??]** with **[??]** parameters, reaching **[??] %** validation accuracy after 20 epochs. **[Explain why it beat the other candidates, e.g. a better balance of width and depth for a short schedule.]** Removing one component at a time (Δ = change in validation accuracy):

* **Shortcut** (Δ = **[??]** pp). Without the identity shortcuts the network trains more slowly, as in 3(b). The effect is **[smaller / larger]** than at depth 56 because this network is only **[??]** layers deep.
* **Augmentation** (Δ = **[??]** pp). Usually the largest effect. Without flip, crop and cutout the training loss falls much lower while the validation accuracy is lower: the network overfits even with 45k images.
* **Weight decay** (Δ = **[??]** pp). In a BN network weight decay keeps the weight norms small and thus the *effective* learning rate high late in training, which acts as a regulariser. Its removal costs **[??]** pp.
* **Label smoothing** (Δ = **[??]** pp). A small effect, as expected, coming from less over-confident logits.

Each ablation is a single run with one seed. Differences smaller than about 0.3 pp are within run-to-run noise, so I only draw conclusions from the larger effects.

### Final evaluation on the test set (performed once)

The final model is the chosen architecture trained with the full recipe above: 20 epochs on the 45k training images, selected using the validation set only. The test set is loaded, normalised with the *training* statistics and used **exactly once**, in the cell below. The overall accuracy and the confusion matrix both come from this single pass.

In [ ]:
# ======== THE ONLY USE OF THE CIFAR-10 TEST SET IN THIS NOTEBOOK ========
cifar_test = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
X_test = normalize(torch.tensor(cifar_test.data).permute(0, 3, 1, 2).float().div(255)).to(device)
Y_test = torch.tensor(cifar_test.targets, dtype=torch.long).to(device)
if FAST_DEV_RUN:
    X_test, Y_test = X_test[:512], Y_test[:512]

final = cand_results[best_name]
final_model = build_t4(final['recipe']).to(device)
final_model.load_state_dict(final['state_dict'])
final_model.eval()
with torch.no_grad(), autocast_ctx():
    test_pred = torch.cat([final_model(xb).argmax(1) for xb, _ in batches(X_test, Y_test, 1000)])
test_acc = (test_pred == Y_test).float().mean().item()

print(f'Final model    : {best_name}, {final["params"]:,} parameters, {T4_EPOCHS} training epochs')
print(f'Validation acc : {100 * final["hist"]["val_acc"][-1]:.2f}%')
print(f'TEST accuracy  : {100 * test_acc:.2f}%')

cm = torch.bincount(Y_test * 10 + test_pred, minlength=100).view(10, 10).cpu().numpy()
per_class = cm.diagonal() / cm.sum(1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.25, 1]})
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(10)); axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[0].set_yticks(range(10)); axes[0].set_yticklabels(CLASSES); axes[0].grid(False)
for i in range(10):
    for j in range(10):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='white' if cm[i, j] > cm.max() / 2 else 'black')
axes[0].set_xlabel('predicted'); axes[0].set_ylabel('true'); axes[0].set_title('Test-set confusion matrix')
axes[1].barh(CLASSES[::-1], 100 * per_class[::-1])
for i, v in enumerate(100 * per_class[::-1]):
    axes[1].text(v + 0.5, i, f'{v:.1f}', va='center', fontsize=8)
axes[1].set_xlabel('test accuracy (%)'); axes[1].set_title('Per-class test accuracy'); axes[1].set_xlim(0, 105)
plt.tight_layout(); plt.show()

**Interpretation (test).** ✏️ The final model reaches **[??] %** test accuracy, **[close to / x pp below]** its validation accuracy of **[??] %**. The small difference shows that choosing the architecture and settings on the validation split did not overfit the validation set noticeably. The confusion matrix shows that most errors are between visually similar classes, above all **cat ↔ dog** **[check]**, and also among the other animals (bird, deer) and between automobile and truck. The vehicle classes (ship, automobile, truck) are recognised best **[check]**.

## 5. Conclusions

✏️ *(Rewrite in your own words once all numbers are filled in.)*

1. **Backpropagation.** The hand-written backward passes of `Linear`, `ReLU`, `Sigmoid` and softmax cross-entropy agree with autograd to machine precision and with centred finite differences to about $10^{-8}$. An MLP trained with them reaches about **[??] %** validation accuracy on flattened CIFAR-10, a result limited by the lack of spatial weight sharing and by overfitting. In a 10-layer sigmoid network the gradient shrinks by orders of magnitude towards the input, whereas ReLU with He initialisation keeps it nearly constant across layers.
2. **Optimizers.** My SGD with momentum and Adam match `torch.optim` to round-off. On an ill-conditioned quadratic, SGD zig-zags and is slowed down by the flat direction, momentum accelerates (and overshoots), and Adam's per-coordinate scaling ignores the conditioning. On a CNN every optimizer has a learning-rate window of about one decade. Momentum moves this window about 10× lower than for plain SGD, and Adam's default $10^{-3}$ worked well without tuning.
3. **Regularisation.** With 5k images, data augmentation (especially flip + crop) helped much more than weight decay or dropout, mixup mainly improved calibration (the validation loss), and combining everything gave the smallest generalisation gap.
4. **Residual connections.** Without shortcuts, adding layers *increases* the training loss (the degradation problem). With identity shortcuts, deeper is better, even though the parameter count is identical.
5. **Best CNN.** A **[??]**-parameter residual network trained for 20 epochs reaches **[??] %** test accuracy. Augmentation and the shortcuts were the most important design choices.

### References
* K. He, X. Zhang, S. Ren, J. Sun. *Deep Residual Learning for Image Recognition.* CVPR 2016.
* K. He, X. Zhang, S. Ren, J. Sun. *Delving Deep into Rectifiers.* ICCV 2015 (He / Kaiming initialisation).
* X. Glorot, Y. Bengio. *Understanding the difficulty of training deep feedforward neural networks.* AISTATS 2010.
* D. P. Kingma, J. Ba. *Adam: A Method for Stochastic Optimization.* ICLR 2015.
* T. DeVries, G. W. Taylor. *Improved Regularization of Convolutional Neural Networks with Cutout.* arXiv:1708.04552, 2017.
* H. Zhang, M. Cisse, Y. N. Dauphin, D. Lopez-Paz. *mixup: Beyond Empirical Risk Minimization.* ICLR 2018.
* D. Balduzzi et al. *The Shattered Gradients Problem: If resnets are the answer, then what is the question?* ICML 2017.

### Appendix: exporting this notebook to PDF
In Colab: *File → Print → Save as PDF* works without any setup (check that wide tables and plots are not cut off). Alternatively, *File → Download → .ipynb* and then, on a machine with Jupyter, run `jupyter nbconvert --to webpdf --allow-chromium-download your_index_a02.ipynb`. Rename the result to `your_index_a02.pdf`.